# Exam Workflow (Classification)

## Quick plan
1. Load data and inspect
2. Define target + features
3. Split train/valid/test
4. Build preprocessing + baseline models
5. Compare with multiple metrics
6. Tune top models
7. Final test evaluation + justification

In [ ]:
# =========================
# 1) IMPORTS
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42

In [ ]:
# =========================
# 2) LOAD DATA
# =========================
DATA_PATH = 'your_dataset.csv'
TARGET_COL = 'target'

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())

In [ ]:
# =========================
# 3) QUICK DATA CHECK
# =========================
print('\nDtypes:')
print(df.dtypes)

print('\nMissing values:')
print(df.isna().sum().sort_values(ascending=False).head(15))

print('\nTarget distribution:')
print(df[TARGET_COL].value_counts(dropna=False))

ax = df[TARGET_COL].value_counts().plot(kind='bar', title='Target Distribution')
ax.set_xlabel(TARGET_COL)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 4) SPLIT FEATURES/TARGET + TRAIN/VALID/TEST
# =========================
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
)

print('Train:', X_train.shape, y_train.shape)
print('Valid:', X_valid.shape, y_valid.shape)
print('Test :', X_test.shape, y_test.shape)

In [ ]:
# =========================
# 5) PREPROCESSING PIPELINE
# =========================
numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols),
    ('cat', categorical_pipe, categorical_cols)
], remainder='drop')

print('Numeric columns:', len(numeric_cols))
print('Categorical columns:', len(categorical_cols))

In [ ]:
# =========================
# 6) BASELINE MODELS
# =========================
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'kNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE),
    'Linear SVM': SVC(kernel='linear', random_state=RANDOM_STATE)
}

baseline_results = {}
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    y_pred = pipe.predict(X_valid)
    baseline_results[name] = {
        'Valid Accuracy': accuracy_score(y_valid, y_pred),
        'Valid Precision': precision_score(y_valid, y_pred, average='weighted', zero_division=0),
        'Valid Recall': recall_score(y_valid, y_pred, average='weighted', zero_division=0),
        'Valid F1': f1_score(y_valid, y_pred, average='weighted', zero_division=0)
    }

baseline_df = pd.DataFrame(baseline_results).T.sort_values('Valid F1', ascending=False)
display(baseline_df.round(4))

In [ ]:
# =========================
# 7) QUICK CROSS-VALIDATION (TOP 2)
# =========================
top2_names = baseline_df.index[:2].tolist()
cv_summary = {}

for name in top2_names:
    pipe = fitted_pipelines[name]
    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=3, scoring='f1_weighted', n_jobs=-1)
    cv_summary[name] = {
        'CV F1 mean': cv_f1.mean(),
        'CV F1 std': cv_f1.std()
    }

cv_df = pd.DataFrame(cv_summary).T.sort_values('CV F1 mean', ascending=False)
display(cv_df.round(4))

In [ ]:
# =========================
# 8) TUNE THE TOP 1 MODEL (FAST GRIDS)
# =========================
best_name = cv_df.index[0]
print('Model selected for tuning:', best_name)

param_grids = {
    'Logistic Regression': {'model__C': [0.1, 1, 10]},
    'kNN': {'model__n_neighbors': [3, 5, 7, 11], 'model__weights': ['uniform', 'distance']},
    'Decision Tree': {'model__max_depth': [5, 10, None], 'model__min_samples_split': [2, 10]},
    'Random Forest': {'model__n_estimators': [100, 200], 'model__max_depth': [10, 20, None]},
    'Linear SVM': {'model__C': [0.1, 1, 10]}
}

best_pipe = fitted_pipelines[best_name]
grid = GridSearchCV(
    estimator=best_pipe,
    param_grid=param_grids[best_name],
    scoring='f1_weighted',
    cv=3,
    n_jobs=-1
)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print('Best CV F1:', round(grid.best_score_, 4))
best_model = grid.best_estimator_

In [ ]:
# =========================
# 9) FINAL TEST EVALUATION
# =========================
y_test_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_rec = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Precision: {test_prec:.4f}')
print(f'Test Recall   : {test_rec:.4f}')
print(f'Test F1       : {test_f1:.4f}')

print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_test_pred))

print('\nClassification Report:')
print(classification_report(y_test, y_test_pred, zero_division=0))